In [ ]:
# [셀 1] 패키지 설치\n# - Whisper, Flask, cloudflared 및 필요한 의존성 설치\n!apt-get -y update\n!apt-get -y install ffmpeg\n!pip -q install --upgrade pip\n!pip -q install openai-whisper flask flask-cors requests\n\n# cloudflared 설치 (Quick Tunnel)\n!wget -q -O cloudflared-linux-amd64.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb\n!dpkg -i cloudflared-linux-amd64.deb >/dev/null\n!cloudflared --version\n\nprint(\"✅ 패키지 설치 완료\")\n

In [ ]:
# [셀 2] Whisper Large-v3 모델 로드\nimport torch\nimport whisper\n\nMODEL_NAME = \"large-v3\"\nDEFAULT_LANGUAGE = \"ko\"\nDEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n\nprint(f\"Whisper 모델 로드 중... model={MODEL_NAME}, device={DEVICE}\")\nmodel = whisper.load_model(MODEL_NAME, device=DEVICE)\nprint(\"✅ Whisper Large-v3 로드 완료\")\n

In [ ]:
# [셀 3] Flask 서버 정의\n# - GET /health\n# - POST /transcribe\nfrom flask import Flask, request, jsonify\nfrom werkzeug.utils import secure_filename\nimport os\nimport tempfile\nimport traceback\n\napp = Flask(__name__)\napp.config[\"JSON_AS_ASCII\"] = False\nALLOWED_EXTENSIONS = {\".mp3\", \".wav\", \".m4a\", \".webm\", \".mp4\", \".mpeg\", \".mpga\"}\n\n\ndef _allowed_file(filename: str) -> bool:\n    ext = os.path.splitext(filename.lower())[1]\n    return ext in ALLOWED_EXTENSIONS\n\n\ndef _normalize_segments(raw_segments):\n    normalized = []\n    for seg in raw_segments or []:\n        normalized.append({\n            \"id\": int(seg.get(\"id\", 0)),\n            \"start\": float(seg.get(\"start\", 0.0)),\n            \"end\": float(seg.get(\"end\", 0.0)),\n            \"text\": str(seg.get(\"text\", \"\")).strip(),\n        })\n    return normalized\n\n\n@app.get(\"/health\")\ndef health():\n    return jsonify({\"status\": \"ok\", \"model\": \"large-v3\"})\n\n\n@app.post(\"/transcribe\")\ndef transcribe():\n    temp_path = None\n    try:\n        if \"file\" not in request.files:\n            return jsonify({\"error\": \"파일 필드(file)가 없습니다.\"}), 400\n\n        uploaded = request.files[\"file\"]\n        if uploaded.filename is None or uploaded.filename.strip() == \"\":\n            return jsonify({\"error\": \"업로드된 파일명이 비어 있습니다.\"}), 400\n\n        filename = secure_filename(uploaded.filename)\n        if not _allowed_file(filename):\n            return jsonify({\"error\": \"지원하지 않는 오디오 형식입니다.\"}), 400\n\n        language = (request.form.get(\"language\") or DEFAULT_LANGUAGE).strip()\n        if not language:\n            language = DEFAULT_LANGUAGE\n\n        with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(filename)[1]) as tmp:\n            uploaded.save(tmp.name)\n            temp_path = tmp.name\n\n        result = model.transcribe(\n            temp_path,\n            language=language,\n            task=\"transcribe\",\n            fp16=torch.cuda.is_available(),\n            verbose=False,\n        )\n\n        return jsonify({\n            \"text\": str(result.get(\"text\", \"\")).strip(),\n            \"segments\": _normalize_segments(result.get(\"segments\", [])),\n            \"language\": result.get(\"language\", language),\n        })\n    except Exception as exc:\n        traceback.print_exc()\n        return jsonify({\"error\": str(exc)}), 500\n    finally:\n        if temp_path and os.path.exists(temp_path):\n            try:\n                os.remove(temp_path)\n            except Exception:\n                pass\n

In [ ]:
# [셀 4] cloudflared 터널 시작 + Flask 서버 실행\n# - 생성된 외부 URL을 명확히 출력\nimport re\nimport signal\nimport subprocess\nimport threading\nimport time\n\nSERVER_PORT = 5000\n\n\ndef start_cloudflared_tunnel(port: int):\n    # 이전 cloudflared 프로세스가 남아 있어도 서버 실행이 죽지 않도록 안전 처리\n    subprocess.run([\"pkill\", \"-f\", \"cloudflared\"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n\n    cmd = [\n        \"cloudflared\",\n        \"tunnel\",\n        \"--url\",\n        f\"http://127.0.0.1:{port}\",\n        \"--no-autoupdate\",\n    ]\n    proc = subprocess.Popen(\n        cmd,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT,\n        text=True,\n        bufsize=1,\n    )\n\n    state = {\"url\": None}\n    pattern = re.compile(r\"https://[-a-z0-9]+\\.trycloudflare\\.com\")\n\n    def _reader():\n        if proc.stdout is None:\n            return\n        for line in proc.stdout:\n            line = line.strip()\n            if not line:\n                continue\n            match = pattern.search(line)\n            if match and state[\"url\"] is None:\n                state[\"url\"] = match.group(0)\n                print(\"\\n====================================================\")\n                print(\"[Colab Tunnel URL] 앱에 아래 주소를 입력하세요\")\n                print(state[\"url\"])\n                print(\"====================================================\\n\")\n\n    t = threading.Thread(target=_reader, daemon=True)\n    t.start()\n\n    timeout_sec = 25\n    start_at = time.time()\n    while time.time() - start_at < timeout_sec and state[\"url\"] is None:\n        if proc.poll() is not None:\n            break\n        time.sleep(0.2)\n\n    if state[\"url\"] is None:\n        print(\"⚠️ 터널 URL 자동 추출에 실패했습니다. 아래 로그를 확인해 주세요.\")\n    return proc\n\n\ncloudflared_proc = start_cloudflared_tunnel(SERVER_PORT)\nprint(f\"Flask 서버 시작: http://0.0.0.0:{SERVER_PORT}\")\nprint(\"헬스체크: /health\")\n\ntry:\n    app.run(host=\"0.0.0.0\", port=SERVER_PORT, debug=False, use_reloader=False)\nfinally:\n    if cloudflared_proc and cloudflared_proc.poll() is None:\n        cloudflared_proc.send_signal(signal.SIGINT)\n        try:\n            cloudflared_proc.wait(timeout=5)\n        except Exception:\n            cloudflared_proc.kill()\n